# Imports

In [10]:
# Imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from torch.nn.parameter import Parameter

# Functions

In [11]:
def generalized_bell(x, p):
    '''
    Calculates Generalized Bell values based on the input data and premise parameters of the ANFIS model.

    Parameters:
    - x (torch.Tensor): Input data tensor (as a batch or a single input).
    - p (torch.Tensor): 3D tensor containing 'A', 'B' and 'C' parameters by data dimension
                        and by ANFIS rule. The parameters represents the shape of the Bell curve,
                        meaning the width, slope and height respectively.

    Returns:
    - torch.Tensor: 2D or 3D tensor with Resulting Generalized Bell values (for single or batch input
                    respectively).

    Explanation:
    This function calculates Generalized Bell values based on the input data `x` and the parameters `p`.
    It ensures that the standard deviation values in `p` are not zero by replacing them with a small
    value (1e-6). The formula used to compute the Generalized Bell values is:

    \[ \frac{1}{1 + \exp\left(-0.5 \cdot \left(\frac{|x - p[:, :, 2}|}{p[:, :, 0]}\right)^2\right)} \]

    Note:
    - If `p[:, :, 0]` is zero, it is replaced with 1e-6 to prevent division by zero.

    '''
    return 1/(1 + torch.pow(torch.abs((x - p[:, :, 2])/torch.where(p[:, :, 0] == 0, torch.tensor(1e-6), p[:, :, 0])), 2*p[:, :, 1]))

<>:2: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_2871/4188964859.py:2: SyntaxWarning: invalid escape sequence '\['
  '''


In [12]:
def gaussian2(x, p):
    '''
    Calculates Gaussian values based on the input data and premise parameters of the ANFIS model 
    (is used as a membership function). It also works with more than one input (batches).

    Parameters:
    - x (torch.Tensor): Input data tensor (as a batch or a single input).
    - p (torch.Tensor): 3D tensor containing 'mu' and 'sigma' parameters by data dimension
                        and by ANFIS rule.

    Returns:
    - torch.Tensor: Tensor with resulting Gaussian values (either as a batch or a single output).

    Explanation:
    This function calculates Gaussian values based on the input data `x` and the parameters `p`.
    It ensures that the standard deviation values in `p` are not zero by replacing them with a small
    value (1e-6). The formula used to compute the Gaussian values is:

    \[ \exp\left(-0.5 \cdot \left(\frac{x - p[:, :, 0]}{p[:, :, 1]}\right)^2\right) \]

    Note:
    - If `p[:, :, 1]` is zero, it is replaced with 1e-6 to prevent division by zero.

    '''
    return torch.exp(-0.5 * torch.pow((x - p[:, :, 0])/torch.where(p[:, :, 1] == 0, torch.tensor(1e-6), p[:, :, 1]), 2))

<>:2: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_2871/4237469877.py:2: SyntaxWarning: invalid escape sequence '\['
  '''


In [13]:
def weighted_linear(x, c, w):
    """
    Compute the weighted linear combination of input features and coefficients (representing the
    consequent parameters of the ANFIS model). It is manufactured in such a way that it also works with
    more than one input (batches) and will be used to calculate the individual outputs of each rule of
    the ANFIS model.

    Parameters:
    - x (torch.Tensor): Input data tensor.
    - c (torch.Tensor): Coefficients tensor for the linear combination (consequent parameters).
    - w (torch.Tensor): Weights tensor for element-wise multiplication.

    Returns:
    - torch.Tensor: Resulting weighted linear combination.

    Explanation:
    This function calculates the weighted linear combination of the input features `x` and the coefficients `c`.
    It involves matrix multiplication of the input features by the transposed coefficients (excluding the last column),
    adding the last column of coefficients, and then element-wise multiplication by the weights `w`.
    The formula used is:

    \[ (x \cdot c[:, :-1].T + c[:, -1]) \cdot w \]

    Note:
    - The weights `w` are applied element-wise.

    """
    return (torch.bmm(x.unsqueeze(0).expand(c[:, :, :-1].size(0), -1, -1), torch.transpose(c[:, :, :-1], 1, 2)) + c[:, :, -1].unsqueeze(1)).mul(w.unsqueeze(0))

<>:2: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_2871/724767595.py:2: SyntaxWarning: invalid escape sequence '\['
  """


# ANFIS

## Structures (for layers)

### Layer 1: Fuzzify

#### Implementation

In [14]:
class FuzzifyLayer(nn.Module):
    '''
    Fuzzification layer of an Adaptive Neuro-Fuzzy Inference System (ANFIS) model.

    Attributes:
    - input_size (int): The size of the input features.
    - dtype (torch.dtype): The data type of the input data (to initialize the premises)
    - mf (function): Membership function to use (default: generalized_bell).
    - params (list): List of parameter names for the membership function (default: ['a', 'b', 'c']).
    - premises (torch.nn.Parameter): Trainable parameters for the fuzzification layer.

    Methods:
    - __init__: Initializes a new FuzzifyLayer instance.
    - init_premises: Initializes the premises based on input training data.
    - forward: Performs a forward pass through the fuzzification layer.
    - premises_structure: Prints the structure of the fuzzy premises as a Pandas dataframe.
    - plot_premises: Plots the fuzzy rules.

    Example Usage:
    >>> input_data = torch.randn((3, 4))
    >>> fuzzify_layer = FuzzifyLayer(input_data, init_rules=3)
    >>> membership_values = fuzzify_layer(input_data)
    '''
    
    
    def __init__(self, x_train, init_fuzzy_rules=1, mf=generalized_bell, mf_params=['a', 'b', 'c'], init_mode=0):
        """
        Initializes a new FuzzifyLayer instance.

        Parameters:
        - x_train (torch.tensor): Input training data.
        - init_rules (int): The number of initial fuzzy rules (default: 1).
        - mf (function): Membership function to use (default: generalized_bell).
        - params (list): List of parameter names for the membership function (default: ['a', 'b', 'c']).
        - init_mode (int): Numeric flag for initializing the fuzzy premises (default: 0, meaning it will be initialized based on the input data,
                           otherwise it will be initialized randomly).

        """
        super(FuzzifyLayer, self).__init__()
        # Input data info
        self.input_size = x_train.shape[1]
        self.dtype = x_train.dtype
        self.max_val = torch.max(x_train).round().item()
        self.min_val = torch.min(x_train).round().item()
        
        self.input_shape = x_train.shape # QUIZAS SEA MAS CONVENIENTE OBTENER INFO DIRECTAMENTE DESDE SHAPE
        
        # Fuzzification parameters
        self.mf = mf
        self.mf_params = mf_params

        # Initialize premises
        self.premises = Parameter(self.init_premises(x_train=x_train, init_fuzzy_rules=init_fuzzy_rules, init_mode=init_mode), requires_grad=True)



    def init_premises(self, x_train, init_fuzzy_rules, init_mode):
        """
        Initializes the fuzzy premises based on input training data.

        Parameters:
        - x_train (torch.Tensor): Training data for initializing fuzzy premises.
        - init_rules (int): The number of initial fuzzy rules.

        Returns:
        - torch.Tensor: Initialized fuzzy premises.

        """
        premises = torch.zeros(self.input_size, init_fuzzy_rules, len(self.mf_params), dtype=self.dtype)
        
        if init_mode != 0:
            premises += 2 * torch.rand(self.input_size, init_fuzzy_rules, len(self.mf_params), dtype=self.dtype) - 1

        elif self.mf == gaussian2:
            if init_fuzzy_rules > 1:
                min = torch.min(x_train, dim=0).values
                max = torch.max(x_train, dim=0).values
                stp = (max - min) / (init_fuzzy_rules - 1)
                for i in range(self.input_size):
                    h = torch.arange(min[i], max[i] + stp[i], stp[i])
                    premises[i, :, 0] = h[:init_fuzzy_rules]
                    premises[i, :, 1] = stp[i]/2
            else:
                for i in range(self.input_size):
                    premises[i, :, 0] = torch.mean(x_train[:, i])
                    premises[i, :, 1] = torch.std(x_train[:, i])

        elif self.mf == generalized_bell:
            if init_fuzzy_rules > 1:
                min = torch.min(x_train, dim=0).values
                max = torch.max(x_train, dim=0).values
                stp = (max - min) / (init_fuzzy_rules - 1)
                for i in range(self.input_size):
                    h = torch.arange(min[i], max[i] + stp[i], stp[i])
                    premises[i, :, 2] = h[:init_fuzzy_rules] # C : centro
                    premises[i, :, 0] = stp[i]/2 # A : ancho
                    premises[i, :, 1] = 8 # B : pendiente
            else:
                for i in range(self.input_size):
                    premises[i, :, 2] = torch.mean(x_train[:, i]) # C : centro
                    premises[i, :, 0] = torch.std(x_train[:, i]) # A : ancho
                    premises[i, :, 1] = 2 # B : pendiente

        return premises



    def forward(self, x):
        """
        Performs a forward pass through the fuzzification layer.

        Parameters:
        - x (torch.Tensor): Input tensor.

        Returns:
        - torch.Tensor: Output tensor (membership values).

        """
        return self.mf(x.unsqueeze(x.dim()), self.premises)



    @property
    def premises_structure(self):
        """
        return the structure of the fuzzy premises on a dataframe.

        """
        df = pd.DataFrame()
        rules = ['Fuzzy rule {}'.format(i) for i in range(1, self.premises.data.shape[1]+1)]
        
        for i in range(self.input_size):
            for j in range(len(self.mf_params)):
                column = pd.Series(self.premises.data[i,:,j], index=rules, name=self.mf_params[j] + f' (x{i})', )
                df[self.mf_params[j] + f' (x{i})'] = column

        return df


    
    @property
    def plot_premises(self):
        """
        plot the fuzzy rules.

        """
        variables = [f'x{i}' for i in range(self.input_size)]
        dataframe = self.premises_structure
        
        sep = round((0.1*(self.max_val - self.min_val))) + 1
        x = np.linspace(self.min_val - sep, self.max_val + sep, 500) 

        fig, axes = plt.subplots(nrows=self.premises.data.shape[1], ncols=len(variables), figsize=(15, 8), sharex=True, sharey=True)

        for i, rule in enumerate(dataframe.index):
            for j, var in enumerate(variables):

                if self.mf == gaussian2:
                    mu = dataframe.loc[rule, f'mu ({var})']
                    sigma = dataframe.loc[rule, f'sigma ({var})']
                    y = np.exp(-((x - mu) ** 2) / (2 * sigma ** 2))

                elif self.mf == generalized_bell:
                    a = dataframe.loc[rule, f'a ({var})']
                    b = dataframe.loc[rule, f'b ({var})']
                    c = dataframe.loc[rule, f'c ({var})']
                    y = 1/(1 + np.power(np.abs((x - c) / a), 2*b))

                ax = axes[i, j]
                ax.plot(x, y, label=f'{rule}, {var}')
                ax.set_title(f'{rule}, {var}')
                ax.grid(True)
                if i == self.premises.data.shape[1] - 1:
                    ax.set_xlabel('x')
                if j == 0:
                    ax.set_ylabel('Membership Value')

        plt.tight_layout()
        plt.show()

        return None

In [15]:
x_train = torch.randn((5,2))
x_train

tensor([[-0.0470, -0.5466],
        [-0.2638,  1.4498],
        [-0.4665, -0.8416],
        [-0.5690,  0.1795],
        [ 0.6857, -0.1900]])

In [17]:
fuzzy_layer = FuzzifyLayer(x_train, init_fuzzy_rules=3)
fuzzy_layer.premises_structure

,a (x0),b (x0),c (x0),a (x1),b (x1),c (x1)
Fuzzy rule 1,0.313675,8.0,-0.568956,0.572825,8.0,-0.841552
Fuzzy rule 2,0.313675,8.0,0.058394,0.572825,8.0,0.304099
Fuzzy rule 3,0.313675,8.0,0.685745,0.572825,8.0,1.449750


In [ ]:
fuzzy_output = fuzzy_layer(x_train)
fuzzy_output

tensor([[[1.7317e-06, 1.0000e+00, 1.8948e-04],
         [2.3283e-10, 1.5259e-05, 1.0000e+00]],

        [[2.3283e-10, 1.5259e-05, 1.0000e+00],
         [1.0510e-06, 1.0000e+00, 3.7963e-04]],

        [[1.0000e+00, 1.5259e-05, 2.3283e-10],
         [9.4212e-06, 1.0000e+00, 2.5086e-05]],

        [[1.0319e-07, 9.9309e-01, 2.2189e-02],
         [1.0000e+00, 1.5259e-05, 2.3283e-10]],

        [[1.4401e-03, 9.9999e-01, 4.4373e-07],
         [4.0830e-05, 1.0000e+00, 6.0370e-06]]], grad_fn=<MulBackward0>)

### Layer 2: Firing Levels

#### r-Implementation

In [25]:
class r_FiringLevelsLayer(nn.Module):
    """
    Class for calculating firing levels in an Adaptive Neuro-Fuzzy Inference System (ANFIS) model.

    Methods:
    - __init__: Initializes a new FiringLevelsLayer instance.
    - forward: Performs a forward pass to calculate firing levels.

    Example Usage:
    >>> firing_levels_layer = FiringLevelsLayer()
    >>> firing_levels = firing_levels_layer(membership_values) #assuming 'membership_values' is the tensor obtained from the Fuzzification Layer

    """
    def __init__(self):
        """
        Initializes a new FiringLevelsLayer instance.

        """
        super(r_FiringLevelsLayer, self).__init__()


    def forward(self, m):
        """
        Performs a forward pass through the layer to calculate firing levels.

        Parameters:
        - m (torch.Tensor): Input tensor containing the membership values for each rule.

        Returns:
        - torch.Tensor: Output tensor (Firing levels).

        """
        w = m.prod(dim=m.dim()-2)
        return w

#### Testing

In [23]:
x_train = torch.randn((4, 2))
x_train

tensor([[-0.8016,  0.1239],
        [-0.1469,  0.4530],
        [-0.6865, -0.3608],
        [-0.5062,  0.9527]])

In [26]:
fuzzy_layer = FuzzifyLayer(x_train, init_rules=3)
fuzzy_layer.premises_structure

,a (x0),b (x0),c (x0),a (x1),b (x1),c (x1)
Fuzzy rule 1,0.16366,8.0,-0.801579,0.328376,8.0,-0.360839
Fuzzy rule 2,0.16366,8.0,-0.474259,0.328376,8.0,0.295914
Fuzzy rule 3,0.16366,8.0,-0.146939,0.328376,8.0,0.952667


In [27]:
fuzzy_output = fuzzy_layer(x_train)
fuzzy_output

tensor([[[1.0000e+00, 1.5259e-05, 2.3283e-10],
         [1.9652e-03, 9.9997e-01, 3.6879e-07]],

        [[2.3283e-10, 1.5259e-05, 1.0000e+00],
         [4.9381e-07, 9.9999e-01, 1.2083e-03]],

        [[9.9642e-01, 1.5434e-02, 5.1403e-09],
         [1.0000e+00, 1.5259e-05, 2.3283e-10]],

        [[7.8950e-05, 1.0000e+00, 3.4372e-06],
         [2.3283e-10, 1.5259e-05, 1.0000e+00]]], grad_fn=<MulBackward0>)

In [28]:
firing_levels_layer = r_FiringLevelsLayer()
firing_levels_output = firing_levels_layer(fuzzy_output)
firing_levels_output

tensor([[1.9652e-03, 1.5258e-05, 8.5867e-17],
        [1.1497e-16, 1.5258e-05, 1.2083e-03],
        [9.9642e-01, 2.3550e-07, 1.1968e-18],
        [1.8382e-14, 1.5259e-05, 3.4372e-06]], grad_fn=<ProdBackward1>)

In [29]:
t = fuzzy_output[0]
t

tensor([[1.0000e+00, 1.5259e-05, 2.3283e-10],
        [1.9652e-03, 9.9997e-01, 3.6879e-07]], grad_fn=<SelectBackward0>)

In [30]:
comb = torch.cartesian_prod(*torch.unbind(t, dim=0)).prod(dim=-1) # todas las combinaciones de membership values multiplicadas
comb

tensor([1.9652e-03, 9.9997e-01, 3.6879e-07, 2.9986e-08, 1.5258e-05, 5.6273e-12,
        4.5755e-13, 2.3282e-10, 8.5867e-17], grad_fn=<ProdBackward1>)

In [31]:
cOMB = torch.cat([torch.cartesian_prod(*torch.unbind(t, dim=0)).prod(dim=-1) for t in fuzzy_output]).reshape(-1, fuzzy_output.shape[-1]**fuzzy_output.shape[-2])
cOMB

tensor([[1.9652e-03, 9.9997e-01, 3.6879e-07, 2.9986e-08, 1.5258e-05, 5.6273e-12,
         4.5755e-13, 2.3282e-10, 8.5867e-17],
        [1.1497e-16, 2.3283e-10, 2.8133e-13, 7.5348e-12, 1.5258e-05, 1.8437e-08,
         4.9381e-07, 9.9999e-01, 1.2083e-03],
        [9.9642e-01, 1.5204e-05, 2.3200e-10, 1.5434e-02, 2.3550e-07, 3.5935e-12,
         5.1403e-09, 7.8434e-14, 1.1968e-18],
        [1.8382e-14, 1.2047e-09, 7.8950e-05, 2.3283e-10, 1.5259e-05, 1.0000e+00,
         8.0029e-16, 5.2447e-11, 3.4372e-06]], grad_fn=<ViewBackward0>)

In [32]:
fuzzy_output[0]

tensor([[1.0000e+00, 1.5259e-05, 2.3283e-10],
        [1.9652e-03, 9.9997e-01, 3.6879e-07]], grad_fn=<SelectBackward0>)

In [33]:
fuzzy_output[0].shape

torch.Size([2, 3])

In [34]:
fuzzy_output.shape

torch.Size([4, 2, 3])

---

#### u-Implementation

In [19]:
class FiringLevelsLayer(nn.Module):
    """
    Class for calculating firing levels in an Adaptive Neuro-Fuzzy Inference System (ANFIS) model.

    Methods:
    - __init__: Initializes a new FiringLevelsLayer instance.
    - forward: Performs a forward pass to calculate firing levels.

    Example Usage:
    >>> firing_levels_layer = FiringLevelsLayer()
    >>> firing_levels = firing_levels_layer(membership_values) #assuming 'membership_values' is the tensor obtained from the Fuzzification Layer

    """
    def __init__(self):
        """
        Initializes a new FiringLevelsLayer instance.

        """
        super(FiringLevelsLayer, self).__init__()


    def forward(self, m):
        """
        Performs a forward pass through the layer to calculate firing levels.

        Parameters:
        - m (torch.Tensor): Input tensor containing the membership values for each rule.

        Returns:
        - torch.Tensor: Output tensor (Firing levels).

        """
        w = torch.cat([torch.cartesian_prod(*torch.unbind(t, dim=0)).prod(dim=-1) for t in m]).reshape(-1, m.shape[-1]**m.shape[-2])
        return w

#### Compute

In [14]:
firing_levels_layer = FiringLevelsLayer()
firing_levels_output = firing_levels_layer(fuzzy_output)
firing_levels_output

tensor([[4.0319e-16, 2.6423e-11, 1.7317e-06, 2.3283e-10, 1.5259e-05, 1.0000e+00,
         4.4116e-14, 2.8912e-09, 1.8948e-04],
        [2.4470e-16, 2.3283e-10, 8.8389e-14, 1.6037e-11, 1.5259e-05, 5.7925e-09,
         1.0510e-06, 1.0000e+00, 3.7963e-04],
        [9.4212e-06, 1.0000e+00, 2.5086e-05, 1.4375e-10, 1.5259e-05, 3.8277e-10,
         2.1935e-15, 2.3283e-10, 5.8407e-15],
        [1.0319e-07, 1.5745e-12, 2.4025e-17, 9.9309e-01, 1.5153e-05, 2.3122e-10,
         2.2189e-02, 3.3857e-07, 5.1663e-12],
        [5.8797e-08, 1.4401e-03, 8.6936e-09, 4.0829e-05, 9.9999e-01, 6.0369e-06,
         1.8117e-11, 4.4373e-07, 2.6788e-12]], grad_fn=<ViewBackward0>)

### Layer 3: Normalization

#### Implementation

In [20]:
class NormalizationLayer(nn.Module):
    """
    Class for normalize the firing levels in an Adaptive Neuro-Fuzzy Inference System (ANFIS) model.

    Methods:
    - __init__: Initializes a new NormalizeLayer instance.
    - forward: Performs a forward pass to normalize the firing levels obtained on a previous layer.

    Example Usage:
    >>> normalization_layer = NormalizationLayer()
    >>> norm_firing_levels = normalization_layer(firing_levels) #assuming 'firing_levels' is the tensor obtained from the Firing Levels Layer

    """
    def __init__(self):
        """
        Initializes a new FiringLevelsLayer instance.

        """
        super(NormalizationLayer, self).__init__()


    def forward(self, w):
        """
        Performs a forward pass through the layer to normalize the firing levels.

        Parameters:
        - x (torch.Tensor): Input tensor containing the membership values for each rule.

        Returns:
        - torch.Tensor: Output tensor (Normalized Firing levels).

        """
        sum = torch.sum(w, dim=-1, keepdim=True)
        sum[sum == 0] = 1
        w = w/sum
        return w

#### Compute

In [16]:
normalization_layer = NormalizationLayer()

In [17]:
normalization_output = normalization_layer(firing_levels_output)
'''
[[~w(rule1), ~w(rule2)],  (of batch input1)
 [~w(rule1), ~w(rule2)],  (of batch input2)
 ...
 [~w(rule1), ~w(rule2)]]  (of last batch input)
'''
normalization_output

tensor([[4.0311e-16, 2.6418e-11, 1.7313e-06, 2.3278e-10, 1.5255e-05, 9.9979e-01,
         4.4107e-14, 2.8906e-09, 1.8944e-04],
        [2.4461e-16, 2.3274e-10, 8.8354e-14, 1.6030e-11, 1.5253e-05, 5.7903e-09,
         1.0506e-06, 9.9960e-01, 3.7948e-04],
        [9.4207e-06, 9.9995e-01, 2.5085e-05, 1.4375e-10, 1.5258e-05, 3.8275e-10,
         2.1934e-15, 2.3282e-10, 5.8404e-15],
        [1.0163e-07, 1.5508e-12, 2.3663e-17, 9.7813e-01, 1.4925e-05, 2.2774e-10,
         2.1855e-02, 3.3347e-07, 5.0884e-12],
        [5.8711e-08, 1.4379e-03, 8.6808e-09, 4.0769e-05, 9.9851e-01, 6.0281e-06,
         1.8091e-11, 4.4308e-07, 2.6749e-12]], grad_fn=<DivBackward0>)

### Layer 4

#### Implementation

In [21]:
class ConsequentLayer(nn.Module):
    """
    Class for representing the fourth layer (consequent layer) of an Adaptive Neuro-Fuzzy Inference System (ANFIS) model.

    Attributes:
    - function (function): Consequent function to use (default: weighted_linear).
    - consequents (torch.nn.Parameter): Trainable parameters for the consequent layer.


    Methods:
    - __init__: Initializes a new ConsequentLayer instance.
    - forward: Performs a forward pass to calculate the consequent layer output.
    - consequents_structure: Prints the structure of the consequent parameters.

    Example Usage:
    >>> input_data = torch.randn((5, 3))  # Assuming input tensor shape (batch_size, num_input_features)
    >>> consequent_layer = ConsequentLayer(input_data.shape[1], input_data.dtype, init_rules=2)
    >>> output = consequent_layer(input_data, weights) # Assuming weight is the tensor obtained from the normalization layer with shape (batch_size, num_rules)

    """
    def __init__(self, input_size, dtype, init_consequent_rules=1, function=weighted_linear, outputs=1):
        """
        Initializes a new ConsequentLayer instance.

        Parameters:
        - input_size (int): Number of input features.
        - dtype (torch.dtype): Data type for the consequents.
        - init_rules (int): Number of initial rules.
        - function (callable): Consequent function to apply.
        - outputs (int): Number of outputs.

        """
        super(ConsequentLayer, self).__init__()
        self.function = function
        self.consequents = Parameter(2 * torch.rand(outputs, init_consequent_rules, input_size + 1, dtype=dtype) - 1, requires_grad=True)


    def forward(self, x, w):
        """
        Performs a forward pass to calculate the consequent layer output.

        Parameters:
        - x (torch.Tensor): Input tensor.
        - w (torch.Tensor): Weights tensor.

        Returns:
        - torch.Tensor: Output tensor (Outputs by rule of the ANFIS model).

        """
        outputs = self.function(x, self.consequents, w)
        return outputs


    @property
    def consequents_structure(self):
        """
        Returns the structure of the consequent parameters on a list of Pandas dataframes.
        Each element of the list represents the consequents of one of the outputs.

        """
        dfs = [pd.DataFrame() for _ in range(self.consequents.data.shape[0])]

        rules = ['rule {}'.format(i) for i in range(1, self.consequents.data.shape[1]+1)]

        for o in range(self.consequents.data.shape[0]):
            for i in range(self.consequents.data.shape[2]):
                name=f'c{i} (x{i})'
                if (i == self.consequents.data.shape[2]-1): name=f'c{i}'
                column = pd.Series(self.consequents.data[o,:,i], index=rules, name=name)
                dfs[o][name] = column

        return dfs

#### Testing

In [21]:
x_train

tensor([[ 0.4120,  1.6349],
        [ 0.9190, -0.0122],
        [-0.2678, -0.3171],
        [ 0.5431, -2.3923],
        [ 0.1788, -0.4988]])

In [29]:
consequent_layer = ConsequentLayer(x_train.shape[1], fuzzy_layer.dtype, init_consequent_rules=9)
consequent_layer.consequents_structure

[         c0 (x0)   c1 (x1)        c2
 rule 1  0.767095  0.614478  0.804057
 rule 2  0.930626  0.271363  0.252684
 rule 3  0.193646  0.936805  0.695979
 rule 4 -0.391917  0.649576  0.722668
 rule 5 -0.279635 -0.753812 -0.808953
 rule 6  0.162200  0.798213  0.364213
 rule 7  0.497474 -0.212814  0.999720
 rule 8  0.277160 -0.052386  0.894243
 rule 9  0.540876 -0.032796  0.425669]

In [30]:
normalization_output

tensor([[4.0311e-16, 2.6418e-11, 1.7313e-06, 2.3278e-10, 1.5255e-05, 9.9979e-01,
         4.4107e-14, 2.8906e-09, 1.8944e-04],
        [2.4461e-16, 2.3274e-10, 8.8354e-14, 1.6030e-11, 1.5253e-05, 5.7903e-09,
         1.0506e-06, 9.9960e-01, 3.7948e-04],
        [9.4207e-06, 9.9995e-01, 2.5085e-05, 1.4375e-10, 1.5258e-05, 3.8275e-10,
         2.1934e-15, 2.3282e-10, 5.8404e-15],
        [1.0163e-07, 1.5508e-12, 2.3663e-17, 9.7813e-01, 1.4925e-05, 2.2774e-10,
         2.1855e-02, 3.3347e-07, 5.0884e-12],
        [5.8711e-08, 1.4379e-03, 8.6808e-09, 4.0769e-05, 9.9851e-01, 6.0281e-06,
         1.8091e-11, 4.4308e-07, 2.6749e-12]], grad_fn=<DivBackward0>)

#### Compute

In [31]:
consequent_output = consequent_layer(x_train, normalization_output)
'''fi = x1*c1i + x2*c2i + c3i, con c1i, c2i, c3i consequent parameters of rule i

[[f1, f2],  (of batch input1)
 [f1, f2],  (of batch input2)
 ...
 [f1, f2]] (of last batch input)
 ]
'''
consequent_output

tensor([[[ 8.5649e-16,  2.8525e-11,  3.9948e-06,  3.7785e-10, -3.2900e-05,
           1.7357e+00,  3.7790e-14,  2.6674e-09,  1.1270e-04],
         [ 3.6728e-16,  2.5708e-10,  7.6206e-14,  5.6843e-12, -1.6118e-05,
           2.9156e-09,  1.5333e-06,  1.1491e+00,  3.5030e-04],
         [ 3.8042e-06, -8.2557e-02,  8.7065e-06,  8.9360e-11, -7.5535e-06,
           2.5907e-11,  2.0486e-15,  1.9478e-10,  1.7009e-15],
         [-2.5342e-08,  1.6889e-13, -3.4074e-17, -1.0213e+00,  1.2574e-05,
          -3.3187e-10,  3.8879e-02,  3.9019e-07,  4.0598e-12],
         [ 3.7262e-08,  4.0791e-04,  2.2856e-09,  1.3396e-05, -4.8220e-01,
          -2.9897e-08,  2.1615e-11,  4.2975e-07,  1.4410e-12]]],
       grad_fn=<MulBackward0>)

### Layer 5

#### Implementation

In [22]:
class OutputLayer(nn.Module):
    """
    Class for representing the last layer (output layer) of an Adaptive Neuro-Fuzzy Inference System (ANFIS) model.

    Methods:
    - __init__: Initializes a new OutputLayer instance.
    - forward: Performs a forward pass to calculate the final output.

    Example Usage:
    >>> output_layer = OutputLayer()
    >>> output = output_layer(rule_outputs) # Assuming rule_outputs is the tensor obtained from the consequent layer with shape (batch_size, rules)

    """
    def __init__(self):
        """
        Initializes a new OutputLayer instance.

        """
        super(OutputLayer, self).__init__()

    def forward(self, x):
        """
        Performs a forward pass to calculate the final output by computing the sum along the last dimension of
        the input tensor.

        Parameters:
        - x (torch.Tensor): Input tensor (Rule outputs).

        Returns:
        - torch.Tensor: Final output.

        """
        return torch.sum(x, dim=-1)

## ANFIS Implementation

In [23]:
class ANFIS(nn.Module):
    """
    Class for representing a type 3 Adaptive Neuro-Fuzzy Inference System (ANFIS).

    Methods:
    - __init__: Initializes a new Type3ANFIS instance.
    - forward: Performs a forward pass through the ANFIS model.
    - intermediate_values: Similar to forward pass but returns the intermediate values obtained by some of the model layers.
    - rules: Returns the number of rules in the system.
    - premises_structure: Prints the structure of the premises.
    - premises: Gets the premises of the fuzzification layer as a tensor.
    - set_premises: Sets the premises parameters of the fuzzification layer.
    - consequents_structure: Prints the structure of the consequents.
    - consequents: Gets the consequents of the consequent layer as a tensor.
    - set_consequents: Sets the consequents parameters of the consequent layer.

    Example Usage:
    >>> input_data = torch.randn((5, 3))  # Assuming input tensor shape (batch_size, num_input_features)
    >>> anfis_model = Type3ANFIS(input_data, init_rules=2)
    >>> output = anfis_model(input_data)

    """


    def __init__(self, x_train, init_fuzzy_rules=1, cf=weighted_linear, mf=generalized_bell, mf_params=['a', 'b', 'c'], init_mode=0, output_type=None, outputs_amount=1):
        """
        Initializes a new Type3ANFIS instance.

        Parameters:
        - x_train (torch.tensor): input training data set.
        - init_rules (int): Number of initial rules (default: 1).
        - cf (callable): Consequent function to apply (default: weighted_linear).
        - mf (callable): Membership function to apply (default: gaussian2).
        - mf_params (list): List of membership function parameters (default: ['mu', 'sigma']).
        - init_mode (int): Numeric flag for initializing the fuzzy premises (default: 0, meaning it will be initialized based on the input data,
                           otherwise it will be initialized randomly).

        """
        super(ANFIS, self).__init__()
        self.input_size = x_train.shape[1]
        self.dtype = x_train.dtype
        self.mf_params = mf_params

        self.fuzzify_layer = FuzzifyLayer(x_train=x_train, init_fuzzy_rules=init_fuzzy_rules, mf=mf, mf_params=mf_params, init_mode=init_mode)
        self.firing_levels_layer = FiringLevelsLayer()
        self.normalization_layer = NormalizationLayer()
        self.consequent_layer = ConsequentLayer(input_size=self.input_size, dtype=self.dtype, init_consequent_rules=init_fuzzy_rules**self.input_size, function=cf, outputs=outputs_amount)
        self.output_layer = OutputLayer()
        self.last_layer = self.def_last_layer(output_type)
            
            
    def def_last_layer(self, output_type):
        """
        Define the last layer of the model.

        Parameters:
        - output_type (str): Type of output of the model.

        """
        last_layer = nn.Identity()
        if (output_type == 'binary'):
            last_layer = nn.Sigmoid()
        elif (output_type == 'multiclass'):
            last_layer = nn.Softmax(dim=0)
        return last_layer


    def forward(self, x):
        """
        Performs a forward pass through the ANFIS model.

        Parameters:
        - x (torch.Tensor): Input tensor.

        Returns:
        - torch.Tensor: Final output.

        """
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.normalization_layer(self.firing_levels_layer(output)))
        output = self.output_layer(output)
        output = self.last_layer(output)
        return output


    def classes_prediction(self, x):
        """
        Emulates a forward pass throught the ANFIS model, but a thereshold for binary output is applied at the end.

        :param x: Input data.
        :type x: torch.Tensor

        :return: Binary predictions.
        :rtype: torch.Tensor

        """
        with torch.no_grad():
            output = self.forward(x)

        if isinstance(self.last_layer, nn.Softmax):
            output = torch.max(output, dim=0).indices

        elif isinstance(self.last_layer, nn.Sigmoid):
            output = (output >= 0.5).to(int)

        return output


    def intermediate_values(self, x):
        """
        Emulates a forward pass throught the ANFIS model, but it returns the intermediate values of the operation.

        Parameters:
        - x (torch.Tensor): Input tensor.

        Returns:
        - w (torch.Tensor): Firing levels.
        - w_norm (torch.Tensor): Normalized firing levels.
        - outputs (torch.Tensor): Outputs by rule of the model

        """
        with torch.no_grad():
          w = self.fuzzify_layer(x)
          w = self.firing_levels_layer(w)
          w_norm = self.normalization_layer(w)
          outputs = self.consequent_layer(x, w_norm)
        return w, w_norm, outputs


    @property
    def rules(self):
        """
        Returns the number of rules in the system.

        """
        return self.consequents.shape[1]


    @property
    def premises_structure(self):
        """
        Prints the structure of the premises.

        """
        return self.fuzzify_layer.premises_structure
    
    @property
    def plot_premises(self):
        """
        plot the fuzzy rules.

        """
        return self.fuzzify_layer.plot_premises


    @property
    def premises(self):
        """
        Return the premises parameters of the fuzzify layer as a torch tensor.

        """
        return self.fuzzify_layer.premises.data


    def set_premises(self, premises):
        """
        Sets the premises of the fuzzification layer.

        Parameters:
        - premises (torch.Tensor): New premises.

        """
        self.fuzzify_layer.premises = Parameter(premises, requires_grad=True)


    @property
    def consequents_structure(self):
        """
        Prints the structure of the consequents.

        """
        return self.consequent_layer.consequents_structure


    @property
    def consequents(self):
        """
        Returns the consequents of the consequent layer as a torch.tensor.

        """
        return self.consequent_layer.consequents.data


    def set_consequents(self, consequents):
        """
        Sets the consequents of the consequent layer.

        Parameters:
        - consequents (torch.Tensor): New consequents.

        """
        self.consequent_layer.consequents = Parameter(consequents, requires_grad=True)

### Testing

In [24]:
x_train = torch.randn((5, 2))
x_train

tensor([[ 0.2947,  0.3000],
        [ 0.1917, -1.9267],
        [-0.0042,  0.8089],
        [ 0.4138,  0.7393],
        [-0.4576, -0.7560]])

In [25]:
test_anfis = ANFIS(x_train, init_fuzzy_rules=3)
test_anfis.premises_structure

,a (x0),b (x0),c (x0),a (x1),b (x1),c (x1)
Fuzzy rule 1,0.217853,8.0,-0.457614,0.683903,8.0,-1.926705
Fuzzy rule 2,0.217853,8.0,-0.021909,0.683903,8.0,-0.558899
Fuzzy rule 3,0.217853,8.0,0.413797,0.683903,8.0,0.808907


In [26]:
test_anfis.consequents_structure

[         c0 (x0)   c1 (x1)        c2
 rule 1  0.586419 -0.278468  0.871268
 rule 2  0.533755  0.315135 -0.823799
 rule 3  0.294414 -0.686397  0.635226
 rule 4 -0.029111 -0.133309  0.371224
 rule 5 -0.434205 -0.691849  0.380044
 rule 6 -0.801341  0.323735 -0.794003
 rule 7  0.739703 -0.309131 -0.632517
 rule 8 -0.486994  0.732742 -0.062862
 rule 9  0.324526  0.991768 -0.600417]

In [27]:
test_anfis(x_train)

tensor([[-0.2035,  0.4038, -0.5288,  0.2671, -1.3059]], grad_fn=<SumBackward1>)